# 02: Feature Extraction (ดึงข้อมูล 15 คน + Baseline Normalization)
ขั้นตอนนี้เราจะดึงข้อมูลของอาสาสมัคร **ทั้ง 15 คน** มาสกัดฟีเจอร์ด้วยหน้าต่าง 15 วินาที
และที่สำคัญที่สุดคือการทำ **Baseline Normalization** (หักลบค่าเฉลี่ยตอนพัก) เพื่อกำจัดความแตกต่างของร่างกายแต่ละบุคคลครับ!

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
from scipy.signal import find_peaks
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded successfully!")

## 1. ดึงข้อมูลและสกัดฟีเจอร์จากทั้ง 15 คน

In [ ]:
# กำหนดรายชื่อ Subject ทั้ง 15 คน (S1 ไม่มีใน Dataset, S12 ถูกข้ามไปในต้นฉบับ)
subjects = [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 13, 14, 15, 16, 17]
data_path = 'data/WESAD/'

window_size_sec = 15
fs = 700
window_points = window_size_sec * fs

features_list = []

for s in subjects:
    file_path = os.path.join(data_path, f'S{s}', f'S{s}.pkl')
    if not os.path.exists(file_path):
        print(f"Skipping S{s}, file not found.")
        continue
        
    print(f"Processing Subject {s}...")
    with open(file_path, 'rb') as f:
        data = pickle.load(f, encoding='latin1')
        
    labels = data['label']
    chest_ecg = data['signal']['chest']['ECG'].flatten()
    chest_eda = data['signal']['chest']['EDA'].flatten()
    chest_temp = data['signal']['chest']['Temp'].flatten()
    chest_acc = data['signal']['chest']['ACC']
    
    for start in range(0, len(labels) - window_points, window_points):
        end = start + window_points
        window_labels = labels[start:end]
        majority_label = np.bincount(window_labels).argmax()
        
        # เอาเฉพาะตอนพัก (1) และ ตอนเครียด (2)
        if majority_label not in [1, 2]:
            continue
            
        ecg_win = chest_ecg[start:end]
        eda_win = chest_eda[start:end]
        temp_win = chest_temp[start:end]
        acc_win = chest_acc[start:end]
        
        # 1. ECG
        peaks, _ = find_peaks(ecg_win, distance=fs/2.5)
        if len(peaks) > 1:
            rr = np.diff(peaks) / fs * 1000
            mean_hr = 60000 / np.mean(rr)
            sdnn = np.std(rr)
            rmssd = np.sqrt(np.mean(np.square(np.diff(rr))))
        else:
            mean_hr, sdnn, rmssd = 0, 0, 0
            
        # 2. EDA
        mean_eda = np.mean(eda_win)
        std_eda = np.std(eda_win)
        
        # 3. Temp
        mean_temp = np.mean(temp_win)
        slope_temp = np.polyfit(np.arange(len(temp_win)), temp_win, 1)[0] * fs
        
        # 4. ACC
        acc_svm = np.sqrt(np.sum(np.square(acc_win), axis=1))
        mean_acc_svm = np.mean(acc_svm)
        var_acc_svm = np.var(acc_svm)
        
        features_list.append({
            'Subject': s,
            'Mean_HR_BPM': mean_hr,
            'SDNN': sdnn,
            'RMSSD': rmssd,
            'Mean_EDA': mean_eda,
            'Std_EDA': std_eda,
            'Mean_Temp': mean_temp,
            'Slope_Temp': slope_temp,
            'Mean_ACC_SVM': mean_acc_svm,
            'Var_ACC_SVM': var_acc_svm,
            'Label': majority_label
        })

df = pd.DataFrame(features_list)
print(f"\nExtraction complete! Total windows: {len(df)}")

## 2. ทำ Baseline Normalization (เคล็ดลับความแม่นยำ 95%)

In [ ]:
# หาค่าเฉลี่ยของตอน "นั่งพัก (Label = 1)" ของแต่ละคน
baseline_means = df[df['Label'] == 1].groupby('Subject')[['Mean_HR_BPM', 'Mean_EDA', 'Mean_Temp', 'Mean_ACC_SVM']].mean()

# วนลูปหักลบค่า Baseline ออกจากข้อมูลทั้งหมดของคนนั้นๆ (สร้างคอลัมน์ใหม่ลงท้ายด้วย _norm)
for s in subjects:
    idx = df['Subject'] == s
    if s in baseline_means.index:
        df.loc[idx, 'Mean_HR_BPM_norm'] = df.loc[idx, 'Mean_HR_BPM'] - baseline_means.loc[s, 'Mean_HR_BPM']
        df.loc[idx, 'Mean_EDA_norm'] = df.loc[idx, 'Mean_EDA'] - baseline_means.loc[s, 'Mean_EDA']
        df.loc[idx, 'Mean_Temp_norm'] = df.loc[idx, 'Mean_Temp'] - baseline_means.loc[s, 'Mean_Temp']
        df.loc[idx, 'Mean_ACC_SVM_norm'] = df.loc[idx, 'Mean_ACC_SVM'] - baseline_means.loc[s, 'Mean_ACC_SVM']

df = df.fillna(0)

# เซฟเป็น v2.csv
df.to_csv('wesad_extracted_features_v2.csv', index=False)
print("Saved features with Baseline Normalization to 'wesad_extracted_features_v2.csv'")
df.head()